In [13]:
#Load the cleaned and merged datasetimport pandas as pd

import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/kathmandu_merged.csv")
df["datetime_utc"] = pd.to_datetime(df["datetime_utc"])
df = df.sort_values(["station", "datetime_utc"]).reset_index(drop=True)

print(df.shape)
df.head()

(13575, 8)


,datetime_utc,pm1,pm25,relativehumidity,temperature,um003,was_missing,station
0,2025-11-25 08:15:00+00:00,23.5,38.9,38.5,21.3,1370.0,False,mid_baneshwor
1,2025-11-25 09:15:00+00:00,27.5,44.9,38.1,21.9,1790.0,False,mid_baneshwor
2,2025-11-25 10:15:00+00:00,29.2,48.9,39.3,21.3,1980.0,False,mid_baneshwor
3,2025-11-25 11:15:00+00:00,29.0,48.0,41.2,20.2,1980.0,False,mid_baneshwor
4,2025-11-25 12:15:00+00:00,29.9,49.4,40.9,20.1,2060.0,False,mid_baneshwor


In [14]:
#Time based features

df["hour"] = df["datetime_utc"].dt.hour
df["day_of_week"] = df["datetime_utc"].dt.dayofweek   # Monday=0 ... Sunday=6
df["month"] = df["datetime_utc"].dt.month

# Nepal's weekend is Saturday; adjust if needed
df["is_weekend"] = df["day_of_week"].isin([5]).astype(int)  # 5 = Saturday and True = 1 and False = 0

df[["datetime_utc", "hour", "day_of_week", "month", "is_weekend"]].head()


,datetime_utc,hour,day_of_week,month,is_weekend
0,2025-11-25 08:15:00+00:00,8,1,11,0
1,2025-11-25 09:15:00+00:00,9,1,11,0
2,2025-11-25 10:15:00+00:00,10,1,11,0
3,2025-11-25 11:15:00+00:00,11,1,11,0
4,2025-11-25 12:15:00+00:00,12,1,11,0


In [15]:
lag_hours = [1, 3, 24, 48]   # lag-1: short-term persistence; lag-3: short-term trend/momentum — is it rising or falling; lag-24: daily cycle (same hour, previous day); lag-48: multi-day persistence — Kathmandu valley temperature inversions can trap pollution for 2+ days

for lag in lag_hours:
    df[f"pm25_lag_{lag}h"] = df.groupby("station")["pm25"].shift(lag)

df[["station", "datetime_utc", "pm25"] + [f"pm25_lag_{lag}h" for lag in lag_hours]].head(10)

,station,datetime_utc,pm25,pm25_lag_1h,pm25_lag_3h,pm25_lag_24h,pm25_lag_48h
0,mid_baneshwor,2025-11-25 08:15:00+00:00,38.9,NaN,NaN,NaN,NaN
1,mid_baneshwor,2025-11-25 09:15:00+00:00,44.9,38.9,NaN,NaN,NaN
2,mid_baneshwor,2025-11-25 10:15:00+00:00,48.9,44.9,NaN,NaN,NaN
3,mid_baneshwor,2025-11-25 11:15:00+00:00,48.0,48.9,38.9,NaN,NaN
4,mid_baneshwor,2025-11-25 12:15:00+00:00,49.4,48.0,44.9,NaN,NaN
5,mid_baneshwor,2025-11-25 13:15:00+00:00,58.5,49.4,48.9,NaN,NaN
6,mid_baneshwor,2025-11-25 14:15:00+00:00,110.0,58.5,48.0,NaN,NaN
7,mid_baneshwor,2025-11-25 15:15:00+00:00,119.0,110.0,49.4,NaN,NaN
8,mid_baneshwor,2025-11-25 16:15:00+00:00,102.0,119.0,58.5,NaN,NaN
9,mid_baneshwor,2025-11-25 17:15:00+00:00,65.0,102.0,110.0,NaN,NaN


In [16]:
rolling_windows = [6, 24]

for window in rolling_windows:
    df[f"pm25_rolling_mean_{window}h"] = (
        df.groupby("station")["pm25"]
        .transform(lambda x: x.rolling(window=window, min_periods=1).mean())
    )
    df[f"pm25_rolling_std_{window}h"] = (
        df.groupby("station")["pm25"]
        .transform(lambda x: x.rolling(window=window, min_periods=1).std())
    )

df[["station", "datetime_utc", "pm25", "pm25_rolling_mean_6h", "pm25_rolling_std_6h",
    "pm25_rolling_mean_24h", "pm25_rolling_std_24h"]].head(10)

,station,datetime_utc,pm25,pm25_rolling_mean_6h,pm25_rolling_std_6h,pm25_rolling_mean_24h,pm25_rolling_std_24h
0,mid_baneshwor,2025-11-25 08:15:00+00:00,38.9,38.900000,NaN,38.900000,NaN
1,mid_baneshwor,2025-11-25 09:15:00+00:00,44.9,41.900000,4.242641,41.900000,4.242641
2,mid_baneshwor,2025-11-25 10:15:00+00:00,48.9,44.233333,5.033223,44.233333,5.033223
3,mid_baneshwor,2025-11-25 11:15:00+00:00,48.0,45.175000,4.520601,45.175000,4.520601
4,mid_baneshwor,2025-11-25 12:15:00+00:00,49.4,46.020000,4.347068,46.020000,4.347068
5,mid_baneshwor,2025-11-25 13:15:00+00:00,58.5,48.100000,6.409056,48.100000,6.409056
6,mid_baneshwor,2025-11-25 14:15:00+00:00,110.0,59.950000,24.939186,56.942857,24.116444
7,mid_baneshwor,2025-11-25 15:15:00+00:00,119.0,72.300000,33.030531,64.700000,31.303400
8,mid_baneshwor,2025-11-25 16:15:00+00:00,102.0,81.150000,32.618016,68.844444,31.811991
9,mid_baneshwor,2025-11-25 17:15:00+00:00,65.0,83.983333,29.777200,68.460000,30.017262


In [17]:
#Set target variable: PM2.5 concentration in the next hour (1-hour ahead prediction)
df["target_pm25_next_1h"] = df.groupby("station")["pm25"].shift(-1)

df[["station", "datetime_utc", "pm25", "target_pm25_next_1h"]].head(10)

,station,datetime_utc,pm25,target_pm25_next_1h
0,mid_baneshwor,2025-11-25 08:15:00+00:00,38.9,44.9
1,mid_baneshwor,2025-11-25 09:15:00+00:00,44.9,48.9
2,mid_baneshwor,2025-11-25 10:15:00+00:00,48.9,48.0
3,mid_baneshwor,2025-11-25 11:15:00+00:00,48.0,49.4
4,mid_baneshwor,2025-11-25 12:15:00+00:00,49.4,58.5
5,mid_baneshwor,2025-11-25 13:15:00+00:00,58.5,110.0
6,mid_baneshwor,2025-11-25 14:15:00+00:00,110.0,119.0
7,mid_baneshwor,2025-11-25 15:15:00+00:00,119.0,102.0
8,mid_baneshwor,2025-11-25 16:15:00+00:00,102.0,65.0
9,mid_baneshwor,2025-11-25 17:15:00+00:00,65.0,71.6


In [18]:
#Station as categorical variable (one-hot encoding)
df = pd.get_dummies(df, columns=["station"], prefix="station")

df.filter(like="station_").head()

,station_mid_baneshwor,station_teku
0,True,False
1,True,False
2,True,False
3,True,False
4,True,False


In [19]:
#making sure that the target variable is not leaking information from the future by checking if the station changes between consecutive rows. If it does, we should not use the target variable for that row since it would be from a different station.
boundary = df[df["station_mid_baneshwor"] != df["station_mid_baneshwor"].shift(-1)]
print(boundary[["datetime_utc", "pm25", "target_pm25_next_1h", "station_mid_baneshwor"]])

                   datetime_utc  pm25  target_pm25_next_1h  \
6782  2026-09-03 22:15:00+00:00  10.0                  NaN   
13574 2026-09-03 22:15:00+00:00   5.4                  NaN   

       station_mid_baneshwor  
6782                    True  
13574                  False  


In [20]:
# False = mid_baneshwor, True = teku

print(df.groupby("station_teku")["pm25"].apply(lambda x: x.isna().mean()))

station_teku
False    0.423264
True     0.347320
Name: pm25, dtype: float64


In [21]:
#Drop rows with missing target values (the last row for each station will have a NaN target since there's no next hour data)

before = len(df)
df = df.dropna(subset=["target_pm25_next_1h"]).reset_index(drop=True)
after = len(df)

print(f"Dropped {before - after} rows with missing target ({(before-after)/before:.1%})")
print(f"Remaining rows: {after}")

Dropped 5232 rows with missing target (38.5%)
Remaining rows: 8343


In [22]:
print(df[df["was_missing"]==False]["pm25"].describe())

count    8212.000000
mean       63.273738
std        45.192810
min         0.000000
25%        28.700000
50%        53.700000
75%        90.000000
max       322.000000
Name: pm25, dtype: float64


In [23]:
print(df.shape)
print(df.isna().sum())
df.head()


(8343, 22)
datetime_utc               0
pm1                       31
pm25                      31
relativehumidity          31
temperature               31
um003                     31
was_missing                0
hour                       0
day_of_week                0
month                      0
is_weekend                 0
pm25_lag_1h               63
pm25_lag_3h              126
pm25_lag_24h             551
pm25_lag_48h             870
pm25_rolling_mean_6h      29
pm25_rolling_std_6h       60
pm25_rolling_mean_24h     20
pm25_rolling_std_24h      42
target_pm25_next_1h        0
station_mid_baneshwor      0
station_teku               0
dtype: int64


,datetime_utc,pm1,pm25,relativehumidity,temperature,um003,was_missing,hour,day_of_week,month,...,pm25_lag_3h,pm25_lag_24h,pm25_lag_48h,pm25_rolling_mean_6h,pm25_rolling_std_6h,pm25_rolling_mean_24h,pm25_rolling_std_24h,target_pm25_next_1h,station_mid_baneshwor,station_teku
0,2025-11-25 08:15:00+00:00,23.5,38.9,38.5,21.3,1370.0,False,8,1,11,...,NaN,NaN,NaN,38.900000,NaN,38.900000,NaN,44.9,True,False
1,2025-11-25 09:15:00+00:00,27.5,44.9,38.1,21.9,1790.0,False,9,1,11,...,NaN,NaN,NaN,41.900000,4.242641,41.900000,4.242641,48.9,True,False
2,2025-11-25 10:15:00+00:00,29.2,48.9,39.3,21.3,1980.0,False,10,1,11,...,NaN,NaN,NaN,44.233333,5.033223,44.233333,5.033223,48.0,True,False
3,2025-11-25 11:15:00+00:00,29.0,48.0,41.2,20.2,1980.0,False,11,1,11,...,38.9,NaN,NaN,45.175000,4.520601,45.175000,4.520601,49.4,True,False
4,2025-11-25 12:15:00+00:00,29.9,49.4,40.9,20.1,2060.0,False,12,1,11,...,44.9,NaN,NaN,46.020000,4.347068,46.020000,4.347068,58.5,True,False


In [24]:
import os
os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/kathmandu_features.csv", index=False)
print(f"Saved {len(df)} rows to ../data/processed/kathmandu_features.csv")

Saved 8343 rows to ../data/processed/kathmandu_features.csv
